# MCP Distillation Flow — Generic Tool-Use Data Synthesis

Uses SDG Hub’s built-in **`mcp_distillation`** flow to generate training data for **any** MCP server:

1. Discover tool schemas from a live MCP endpoint  
2. Frontier agent **explores** the server (real tool calls)  
3. Teacher LLM **synthesizes + filters** questions  
4. Frontier agent produces **expert trajectories**  
5. Format traces → SFT `messages` JSONL  

```text
Explore → Diversify → Question synth → Quality filter
       → Expert trajectories → Response filter → Training JSONL
```

**Unlike** `test_mcp_flows.ipynb` (hard-coded Order Ops plans / seed questions), this path only needs:
- MCP server URL  
- Teacher LLM (`.env`)  
- Langflow (or LangGraph) agent wired to the **same** MCP server  

**Prerequisites**
- Writable `.venv` kernel with `sdg_hub`, `fastmcp`, `mcp>=1.8`
- MCP server reachable (Order Ops sample or your own)
- Langflow agent with frontier model + MCP tools enabled

## 0. Install (once per kernel)

In [ ]:
# Use the writable .venv kernel (not read-only app-root).
%pip install -q "mcp>=1.8.0,<2.0.0" fastmcp nest_asyncio pandas python-dotenv
%pip install -q "git+https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub.git"

# Restart kernel after first install, then continue.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
import socket
import sys
import threading
import time
from pathlib import Path
from typing import Any

import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

nest_asyncio.apply()

SAMPLE_DIR = Path.cwd().resolve()
for candidate in [SAMPLE_DIR, *SAMPLE_DIR.parents]:
    if (candidate / "mcp_server.py").exists():
        SAMPLE_DIR = candidate
        break

os.chdir(SAMPLE_DIR)
sys.path.insert(0, str(SAMPLE_DIR))
load_dotenv(SAMPLE_DIR / ".env")

MCP_SERVER_URL = os.environ.get("MCP_SERVER_URL", "http://127.0.0.1:8008/mcp")
MCP_HOST, MCP_PORT = "127.0.0.1", int(os.environ.get("PORT", "8008"))

TEACHER_MODEL = (
    os.environ.get("MODEL")
    or os.environ.get("TEACHER_MODEL_NAME")
    or "openai/gpt-4o"
)
TEACHER_API_BASE = os.environ.get("API_BASE") or os.environ.get("TEACHER_MODEL_URL")
TEACHER_API_KEY = (
    os.environ.get("OPENAI_API_KEY")
    or os.environ.get("API_KEY")
    or os.environ.get("TEACHER_API_KEY")
)

# Frontier agent that can call the MCP server (required by distillation AgentBlocks)
LANGFLOW_URL = os.environ.get(
    "LANGFLOW_URL",
    "http://localhost:7860/api/v1/run/<your-flow-id>",
)
LANGFLOW_API_KEY = os.environ.get("LANGFLOW_API_KEY")
AGENT_FRAMEWORK = os.environ.get("AGENT_FRAMEWORK", "langflow")  # or langgraph

OUT_JSONL = SAMPLE_DIR / "distillation_dataset.jsonl"
OUT_PARQUET = SAMPLE_DIR / "distillation_dataset.parquet"

print(f"SAMPLE_DIR      : {SAMPLE_DIR}")
print(f"MCP_SERVER_URL  : {MCP_SERVER_URL}")
print(f"Teacher         : {TEACHER_MODEL}")
print(f"API base        : {TEACHER_API_BASE or '(default)'}")
print(f"API key set     : {bool(TEACHER_API_KEY)}")
print(f"Agent framework : {AGENT_FRAMEWORK}")
print(f"Langflow URL    : {LANGFLOW_URL}")
print(f"sys.executable  : {sys.executable}")

## 2. Start sample MCP server (optional)

Skip if `MCP_SERVER_URL` already points at your own running server.

In [ ]:
def port_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0


_mcp_thread = globals().get("_mcp_thread")

if port_open(MCP_HOST, MCP_PORT):
    print(f"MCP already listening on {MCP_HOST}:{MCP_PORT}")
elif (SAMPLE_DIR / "mcp_server.py").exists():
    import mcp_server as order_ops

    def _run() -> None:
        order_ops.mcp.run(transport="http", host="0.0.0.0", port=MCP_PORT)

    _mcp_thread = threading.Thread(target=_run, name="mcp-server", daemon=True)
    _mcp_thread.start()
    for _ in range(60):
        if port_open(MCP_HOST, MCP_PORT):
            break
        time.sleep(0.25)
    else:
        raise RuntimeError("Failed to start sample mcp_server.py")
    print(f"Started sample Order Ops MCP on {MCP_SERVER_URL}")
else:
    raise RuntimeError(
        f"Nothing listening on {MCP_HOST}:{MCP_PORT} and mcp_server.py not found. "
        "Start your MCP server or set MCP_SERVER_URL."
    )

## 3. Discover tools from the MCP server (generic)

No hard-coded tool names — schemas come from `list_tools()` on whatever server you point at.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


async def fetch_mcp_tool_list(url: str) -> tuple[list[dict[str, Any]], str, str]:
    """Return (tool_list, server_name, server_description) from a live MCP server."""
    async with streamablehttp_client(url) as (read, write, _):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            tools_resp = await session.list_tools()

            tool_list: list[dict[str, Any]] = []
            for t in tools_resp.tools:
                schema = getattr(t, "inputSchema", None) or {}
                tool_list.append(
                    {
                        "name": t.name,
                        "description": t.description or "",
                        "inputSchema": schema,
                    }
                )

            server_name = (
                getattr(getattr(init, "serverInfo", None), "name", None)
                or "MCP Server"
            )
            server_description = (
                getattr(getattr(init, "serverInfo", None), "instructions", None)
                or getattr(init, "instructions", None)
                or f"MCP server exposing {len(tool_list)} tools."
            )
            return tool_list, str(server_name), str(server_description)


tool_list, mcp_server_name, mcp_server_description = await fetch_mcp_tool_list(
    MCP_SERVER_URL
)

print(f"Server : {mcp_server_name}")
print(f"Desc   : {mcp_server_description[:120]}")
print(f"Tools  : {[t['name'] for t in tool_list]}")
assert tool_list, "MCP server returned zero tools"

df_in = pd.DataFrame(
    {
        "tool_list": [tool_list],
        "mcp_server_name": [mcp_server_name],
        "mcp_server_description": [mcp_server_description],
    }
)
display(df_in.assign(tool_list=df_in["tool_list"].map(lambda x: f"{len(x)} tools")))

## 4. Load the MCP distillation flow

Flow ID `new-night-835` is the packaged **MCP Server Distillation** pipeline in SDG Hub.

In [ ]:
from sdg_hub import Flow, FlowRegistry

FlowRegistry.discover_flows()

FLOW_ID = "new-night-835"  # MCP Server Distillation
flow_path = FlowRegistry.get_flow_path(FLOW_ID)
if flow_path is None:
    # Fallback: resolve from installed sdg_hub package
    import sdg_hub

    pkg = Path(sdg_hub.__file__).resolve().parent
    flow_path = str(pkg / "flows" / "agentic" / "mcp_distillation" / "flow.yaml")

flow = Flow.from_yaml(flow_path)
print(f"Flow     : {flow.metadata.name}")
print(f"Version  : {flow.metadata.version}")
print(f"Blocks   : {len(flow.blocks)}")
print(f"Agents   : {flow._detect_agent_blocks()}")
print(f"Path     : {flow_path}")
flow.print_info()

## 5. Configure teacher LLM + frontier agent

| Role | Config | Purpose |
| --- | --- | --- |
| **Teacher** | `set_model_config` | Question synthesis + quality scoring (LiteLLM) |
| **Frontier agent** | `set_agent_config` | Explore MCP + solve questions via real tool calls |

Langflow agent must use a **frontier** model connected to this MCP server — not the small student model you will fine-tune later.

In [ ]:
if not TEACHER_API_KEY:
    raise RuntimeError("Set TEACHER_API_KEY (or OPENAI_API_KEY) in .env")

if "<your-flow-id>" in LANGFLOW_URL:
    raise RuntimeError(
        "Set LANGFLOW_URL in .env to your Langflow run endpoint, e.g.\n"
        "  LANGFLOW_URL=http://localhost:7860/api/v1/run/<flow-id>\n"
        "The agent must be connected to the same MCP server as MCP_SERVER_URL."
    )

model_kwargs: dict[str, Any] = {
    "model": TEACHER_MODEL,
    "api_key": TEACHER_API_KEY,
}
if TEACHER_API_BASE:
    model_kwargs["api_base"] = TEACHER_API_BASE

flow.set_model_config(**model_kwargs)
print(f"Teacher configured: {TEACHER_MODEL}")

agent_kwargs: dict[str, Any] = {
    "agent_framework": AGENT_FRAMEWORK,
    "agent_url": LANGFLOW_URL,
}
if LANGFLOW_API_KEY:
    agent_kwargs["agent_api_key"] = LANGFLOW_API_KEY

flow.set_agent_config(**agent_kwargs)
# Exploration calls many tools — allow a longer timeout
flow.set_agent_config(timeout=300, blocks=["explore_server"])
print(f"Agent configured: {AGENT_FRAMEWORK} → {LANGFLOW_URL}")

## 6. Run distillation

Default YAML multiplies ×10 candidates. For a first run, override to a smaller volume.

In [ ]:
RUNTIME_PARAMS = {
    # Fewer candidates for a smoke / demo run; raise for production datasets
    "multiply_tool_rows": {"num_samples": 4},
    "sample_tools": {"num_samples": 2},  # tools per question combo
}

print(f"Input rows : {len(df_in)}")
print(f"Tools      : {len(df_in['tool_list'].iloc[0])}")
print(f"Runtime    : {RUNTIME_PARAMS}")

result = flow.generate(df_in, runtime_params=RUNTIME_PARAMS)

if hasattr(result, "to_pandas"):
    df_out = result.to_pandas()
else:
    df_out = result

print(f"\nGenerated {len(df_out)} rows")
print(f"Columns: {list(df_out.columns)}")
display(df_out.head())

## 7. Inspect exploration + quality

In [ ]:
if "server_understanding" in df_out.columns:
    print("=== server_understanding (truncated) ===")
    print(str(df_out["server_understanding"].iloc[0])[:1500])

for col in ["question_quality_rating", "completeness_rating"]:
    if col in df_out.columns:
        print(f"\n{col}:")
        print(df_out[col].value_counts().to_string())

q_col = next(
    (c for c in df_out.columns if "question" in c.lower() and "quality" not in c.lower()),
    None,
)
if q_col:
    print(f"\nSample questions ({q_col}):")
    for q in df_out[q_col].head(5).tolist():
        print(" -", str(q)[:160])

## 8. Format SFT messages and save

In [ ]:
from sdg_hub.core.utils.message_formatter import tool_trace_to_messages

TRACE_COL = "extract_agent_text_tool_trace"
if TRACE_COL not in df_out.columns:
    # Fallback: find a tool-trace-like column
    candidates = [c for c in df_out.columns if "trace" in c.lower()]
    raise KeyError(
        f"Expected column {TRACE_COL!r}. Available trace-like columns: {candidates}\n"
        f"All columns: {list(df_out.columns)}"
    )

# Peek first trace
trace0 = df_out[TRACE_COL].iloc[0]
print(f"First trace steps: {len(trace0) if trace0 is not None else 0}")
if isinstance(trace0, list):
    for i, step in enumerate(trace0[:12]):
        role = step.get("role")
        if role == "assistant" and step.get("tool_calls"):
            for tc in step["tool_calls"]:
                name = (tc.get("function") or {}).get("name", "")
                print(f"  [{i}] tool_call → {name}()")
        elif role == "tool":
            print(f"  [{i}] tool result [{step.get('name', '')}]")
        elif role == "assistant":
            print(f"  [{i}] assistant: {str(step.get('content', ''))[:80]}")

records = []
for _, row in df_out.iterrows():
    trace = row[TRACE_COL]
    tools = row["tool_list"]
    if not isinstance(trace, list) or not trace:
        continue
    msgs = tool_trace_to_messages(trace, tools)
    records.append(
        {
            "messages": msgs,
            "mode": "mcp_distillation",
            "mcp_server_name": row.get("mcp_server_name", mcp_server_name),
            "teacher_model": TEACHER_MODEL,
        }
    )

with OUT_JSONL.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

df_out.to_parquet(OUT_PARQUET, index=False)

print(f"\nWrote {len(records)} SFT trajectories → {OUT_JSONL.name}")
print(f"Wrote full pipeline table     → {OUT_PARQUET.name}")
assert records, "No trajectories formatted — check agent/tool traces"

## 9. Point at *your* MCP server

No code changes to the flow — only env + input discovery:

```bash
# .env
MCP_SERVER_URL=https://your-mcp.example.com/mcp
TEACHER_MODEL_NAME=openai/gpt-4o
TEACHER_MODEL_URL=https://.../v1
TEACHER_API_KEY=sk-...
LANGFLOW_URL=http://localhost:7860/api/v1/run/<flow-id>
LANGFLOW_API_KEY=...
```

Scale up:

```python
result = flow.generate(
    df_in,
    runtime_params={
        "multiply_tool_rows": {"num_samples": 50},
        "sample_tools": {"num_samples": 3},
    },
)
```

Multiple servers: one DataFrame row per server (`tool_list`, name, description).